# Stage-Specific Inflection Points of Waiting Time and Survival in Oral Cavity Cancer
## Supplementary Material: Analytical Code and Reproducibility Protocol

**Associated manuscript:** *Stage-Specific Inflection Points of Waiting Time and Survival in Oral Cavity Cancer: A Population-Based Cohort Study* (submitted to *Frontiers in Oncology*).

**Authors:** Araújo Dias F, De Cicco R, Pereira de Souza R, Briaunyz Milan T.
**Data source:** Hospital-based cancer registry of the Fundação Oncocentro de São Paulo (FOSP), State of São Paulo, Brazil.
**Repository:** https://github.com/Faddragon/Inflection-Points-of-Waiting-Time

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Faddragon/Inflection-Points-of-Waiting-Time/blob/main/Supplementary_Notebook_Oral_Cancer_Inflection_Points_Academic.ipynb)

---

### Objective
To determine, for oral cavity squamous cell carcinoma, the waiting-time cutpoint (*inflection point*) that maximizes the separation of survival curves, for **overall survival (OS)** and **recurrence-free survival (RFS)**, in the whole cohort and stratified by AJCC 7th edition clinical stage (I–IV).

### Waiting-time variables
| Symbol | Dataset column | Definition | Unit |
|---|---|---|---|
| $t_a$ | `CONSDIAG` | Histopathological diagnosis to first specialist consultation | days |
| $t_b$ | `TRATCONS` | First specialist consultation to treatment initiation | days |
| $t_c$ | `DIAGTRAT` | Histopathological diagnosis to treatment initiation ($t_c = t_a + t_b$) | days |

### Outcome definitions (as implemented in Section 2)
| Outcome | Time origin | End of follow-up | Event |
|---|---|---|---|
| OS | Diagnosis (`DTDIAG_CLEAN`) | Last information (`DTULTINFO_CLEAN`) | Death (`EVENT_DEATH`) |
| RFS | Diagnosis (`DTDIAG_CLEAN`) | Recurrence date (`DTRECIDIVA_CLEAN`) if `HAS_RECURRENCE = 1`; otherwise last information | Recurrence (`HAS_RECURRENCE`) |

Follow-up times are converted to months (days / 30.44). Records with non-positive follow-up are excluded, and the OS and RFS cohorts are restricted to the same patients.
*Implementation note:* in this code, a death without a recorded recurrence is censored in the RFS analysis, i.e. it is not scored as an RFS event.

### Analytical workflow
1. Computational environment and dependencies.
2. Data ingestion and cohort construction.
3. Baseline characteristics and distribution of waiting times.
4. Kaplan–Meier estimates of OS and RFS by clinical stage.
5. Multivariable Cox proportional hazards regression.
6. Grid search for inflection points by maximally selected log-rank statistics (overall cohort).
7. Stage-specific inflection points with Benjamini–Hochberg false discovery rate (FDR) adjustment.
8. Export of tables.

### Statistical notes
- **Cutpoint search.** Each candidate cutpoint $t$ splits patients into $\le t$ and $> t$; the inflection point is the $t$ that maximizes the log-rank statistic $Z^2$ (Lausen and Schumacher, 1992).
- **P-values.** The p-value reported at the optimal cutpoint is the *nominal* log-rank p-value (chi-square, 1 degree of freedom) of the selected split. It is **not adjusted for the selection of the maximum** over the search grid (see Altman et al., 1994; Hothorn and Lausen, 2003), and should be interpreted accordingly.
- **Multiplicity.** Benjamini–Hochberg FDR correction is applied in Section 7 across all stage-specific tests (stage × interval × outcome). The Cox model in Section 5 reports unadjusted p-values.
- **Proportional hazards.** The proportional-hazards assumption of the Cox models is not formally tested in this notebook.
- **Interpretation.** Results are observational associations, subject to immortal time bias, severity bias and confounding by indication; they should not be read as individual safe deadlines.

### References
1. Lausen B, Schumacher M. Maximally selected rank statistics. *Biometrics*. 1992;48(1):73–85.
2. Altman DG, Lausen B, Sauerbrei W, Schumacher M. Dangers of using "optimal" cutpoints in the evaluation of prognostic factors. *J Natl Cancer Inst*. 1994;86(11):829–835.
3. Hothorn T, Lausen B. On the exact distribution of maximally selected rank statistics. *Comput Stat Data Anal*. 2003;43(2):121–137.
4. Benjamini Y, Hochberg Y. Controlling the false discovery rate: a practical and powerful approach to multiple testing. *J R Stat Soc Series B*. 1995;57(1):289–300.
5. Davidson-Pilon C. lifelines: survival analysis in Python. *J Open Source Softw*. 2019;4(40):1317.

## 1. Computational Environment and Dependencies
Survival analysis is performed with `lifelines`; multiple-testing correction with `statsmodels`; spreadsheet input and output with `openpyxl`. Package versions should be recorded for each execution (see the version printout at the end of the cell below).

In [ ]:
# Section 1. Computational environment and dependencies
# Install lifelines (survival analysis), statsmodels (FDR correction) and openpyxl (Excel I/O)
!pip install lifelines statsmodels openpyxl -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from statsmodels.stats.multitest import multipletests
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

# Figure style: colorblind-safe palette
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.linewidth"] = 0.8
warnings.filterwarnings('ignore')

# Record package versions for reproducibility
import sys, lifelines, statsmodels, matplotlib, seaborn
print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"lifelines {lifelines.__version__} | statsmodels {statsmodels.__version__} | "
      f"matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")

## 2. Data Ingestion and Cohort Construction
The analytic dataset is read from `data/df_oral_so_curativo.xlsx` in the repository (or downloaded from GitHub if the file is not present locally). The notebook requires the following columns:

| Column | Content |
|---|---|
| `DTDIAG_CLEAN`, `DTCONSULT_CLEAN`, `DTTRAT_CLEAN`, `DTULTINFO_CLEAN`, `DTRECIDIVA_CLEAN` | Dates of diagnosis, consultation, treatment, last information and recurrence |
| `EVENT_DEATH`, `HAS_RECURRENCE` | Death indicator and recurrence indicator (0/1) |
| `CONSDIAG`, `TRATCONS`, `DIAGTRAT` | Waiting times $t_a$, $t_b$, $t_c$ (days) |
| `IDADE`, `SEXO`, `ECGRUP` | Age (years), sex (`1` = male), AJCC 7th edition clinical stage (I–IV) |

Follow-up times for OS and RFS are derived, the two cohorts are restricted to the same patients, and the resulting sample size and event counts are printed for verification.

In [ ]:
# Section 2. Data ingestion and cohort construction
# Path to the analytic dataset (Excel); adjust as needed
import os
REPO_RAW_URL = 'https://raw.githubusercontent.com/Faddragon/Inflection-Points-of-Waiting-Time/main/'
DATA_PATH = 'data/df_oral_so_curativo.xlsx'
if not os.path.exists(DATA_PATH):      # e.g. notebook opened in Colab from GitHub without cloning
    DATA_PATH = REPO_RAW_URL + DATA_PATH

# Load the dataset
df = pd.read_excel(DATA_PATH)
print(f"Initial raw records loaded: {len(df)}")

# Parse date fields
date_cols = ['DTDIAG_CLEAN', 'DTCONSULT_CLEAN', 'DTTRAT_CLEAN', 'DTULTINFO_CLEAN', 'DTRECIDIVA_CLEAN']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# -----------------------------------------------------
# Overall survival (OS): time origin = diagnosis; event = death
# -----------------------------------------------------
df_surv_os = df.copy()
df_surv_os['TIME_TO_EVENT_OS_DAYS'] = (df_surv_os['DTULTINFO_CLEAN'] - df_surv_os['DTDIAG_CLEAN']).dt.days
df_surv_os['TIME_MONTHS_OS'] = df_surv_os['TIME_TO_EVENT_OS_DAYS'] / 30.44
df_surv_os['EVENT_OS'] = df_surv_os['EVENT_DEATH']
df_surv_os = df_surv_os[df_surv_os['TIME_MONTHS_OS'] > 0].copy()

# -----------------------------------------------------
# Recurrence-free survival (RFS): time origin = diagnosis; event = recurrence
# -----------------------------------------------------
df_surv_rfs = df.copy()
df_surv_rfs['TIME_TO_EVENT_RFS_DAYS'] = df_surv_rfs.apply(
    lambda row: (row['DTRECIDIVA_CLEAN'] - row['DTDIAG_CLEAN']).days
    if pd.notna(row['DTRECIDIVA_CLEAN']) and row['HAS_RECURRENCE'] == 1
    else (row['DTULTINFO_CLEAN'] - row['DTDIAG_CLEAN']).days,
    axis=1
)
df_surv_rfs['TIME_MONTHS_RFS'] = df_surv_rfs['TIME_TO_EVENT_RFS_DAYS'] / 30.44
df_surv_rfs['EVENT_RECURRENCE'] = df_surv_rfs['HAS_RECURRENCE']
df_surv_rfs = df_surv_rfs[df_surv_rfs['TIME_MONTHS_RFS'] > 0].copy()

# Restrict OS and RFS analyses to the same patients
common_indices = set(df_surv_os.index).intersection(set(df_surv_rfs.index))
df_surv_os = df_surv_os.loc[list(common_indices)].copy()
df_surv_rfs = df_surv_rfs.loc[list(common_indices)].copy()

print(f"Cohort synchronized successfully: N = {len(df_surv_os)} patients")
print(f"Total Deaths (OS Events): {df_surv_os['EVENT_OS'].sum()} ({df_surv_os['EVENT_OS'].mean()*100:.1f}%)")
print(f"Total Recurrences (RFS Events): {df_surv_rfs['EVENT_RECURRENCE'].sum()} ({df_surv_rfs['EVENT_RECURRENCE'].mean()*100:.1f}%)")

## 3. Baseline Characteristics and Distribution of Waiting Times
Descriptive statistics of the analytic cohort: age, sex and clinical stage, followed by the mean, median and standard deviation of each waiting-time interval.

In [ ]:
# Section 3. Baseline characteristics and distribution of waiting times
# Demographic and clinical summary
summary_table = pd.DataFrame({
    'Variable': ['Age (years, mean ± SD)', 'Sex: Male (%)', 'Stage I (%)', 'Stage II (%)', 'Stage III (%)', 'Stage IV (%)'],
    'Value': [
        f"{df_surv_os['IDADE'].mean():.1f} ± {df_surv_os['IDADE'].std():.1f}",
        f"{(df_surv_os['SEXO'] == 1).sum()} ({(df_surv_os['SEXO'] == 1).mean()*100:.1f}%)",
        f"{(df_surv_os['ECGRUP'] == 'I').sum()} ({(df_surv_os['ECGRUP'] == 'I').mean()*100:.1f}%)",
        f"{(df_surv_os['ECGRUP'] == 'II').sum()} ({(df_surv_os['ECGRUP'] == 'II').mean()*100:.1f}%)",
        f"{(df_surv_os['ECGRUP'] == 'III').sum()} ({(df_surv_os['ECGRUP'] == 'III').mean()*100:.1f}%)",
        f"{(df_surv_os['ECGRUP'] == 'IV').sum()} ({(df_surv_os['ECGRUP'] == 'IV').mean()*100:.1f}%)"
    ]
})
display(summary_table)

# Distribution of waiting times (days)
time_metrics = pd.DataFrame({
    'Interval': ['t_a (Diagnosis to Consult)', 't_b (Consult to Treatment)', 't_c (Diagnosis to Treatment)'],
    'Column': ['CONSDIAG', 'TRATCONS', 'DIAGTRAT'],
    'Mean (Days)': [df_surv_os['CONSDIAG'].mean(), df_surv_os['TRATCONS'].mean(), df_surv_os['DIAGTRAT'].mean()],
    'Median (Days)': [df_surv_os['CONSDIAG'].median(), df_surv_os['TRATCONS'].median(), df_surv_os['DIAGTRAT'].median()],
    'SD': [df_surv_os['CONSDIAG'].std(), df_surv_os['TRATCONS'].std(), df_surv_os['DIAGTRAT'].std()]
})
display(time_metrics.round(1))

## 4. Kaplan–Meier Estimates of OS and RFS by Clinical Stage
Survival functions are estimated with the Kaplan–Meier method for each AJCC stage and compared with the multivariate log-rank test.

In [ ]:
# Section 4. Kaplan-Meier estimates of OS and RFS by clinical stage
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
stages = ['I', 'II', 'III', 'IV']
palette = sns.color_palette('colorblind')
kmf = KaplanMeierFitter()

# Panel A: overall survival
for i, stage in enumerate(stages):
    subset = df_surv_os[df_surv_os['ECGRUP'] == stage]
    kmf.fit(subset['TIME_MONTHS_OS'], event_observed=subset['EVENT_OS'], label=f'Stage {stage} (n={len(subset)})')
    kmf.plot_survival_function(ax=ax1, ci_show=False, color=palette[i], lw=2)

res_os = multivariate_logrank_test(df_surv_os['TIME_MONTHS_OS'], df_surv_os['ECGRUP'], df_surv_os['EVENT_OS'])
ax1.set_title('Overall Survival (OS) by Clinical Stage', fontsize=13, weight='bold')
ax1.set_xlabel('Follow-up Time (Months)')
ax1.set_ylabel('Overall Survival Probability')
ax1.text(0.05, 0.08, f'Multivariate Log-Rank p < 0.001\n(Test Statistic: {res_os.test_statistic:.1f})', 
         transform=ax1.transAxes, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Panel B: recurrence-free survival
for i, stage in enumerate(stages):
    subset = df_surv_rfs[df_surv_rfs['ECGRUP'] == stage]
    kmf.fit(subset['TIME_MONTHS_RFS'], event_observed=subset['EVENT_RECURRENCE'], label=f'Stage {stage} (n={len(subset)})')
    kmf.plot_survival_function(ax=ax2, ci_show=False, color=palette[i], lw=2)

res_rfs = multivariate_logrank_test(df_surv_rfs['TIME_MONTHS_RFS'], df_surv_rfs['ECGRUP'], df_surv_rfs['EVENT_RECURRENCE'])
ax2.set_title('Recurrence-Free Survival (RFS) by Clinical Stage', fontsize=13, weight='bold')
ax2.set_xlabel('Follow-up Time (Months)')
ax2.set_ylabel('Recurrence-Free Probability')
ax2.text(0.05, 0.08, f'Multivariate Log-Rank p < 0.001\n(Test Statistic: {res_rfs.test_statistic:.1f})', 
         transform=ax2.transAxes, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

## 5. Multivariable Cox Proportional Hazards Regression
Cox models for OS and RFS estimate the adjusted association of the waiting times $t_a$ and $t_b$ with each outcome, with adjustment for age, sex and clinical stage (stage I as reference). Hazard ratios are reported per additional day of waiting, with 95% confidence intervals and unadjusted p-values.

In [ ]:
# Section 5. Multivariable Cox proportional hazards regression
# Covariates: t_a, t_b, age, sex, clinical stage (stage I = reference)
def fit_cox_model(df_target, duration_col, event_col):
    features = [duration_col, event_col, 'CONSDIAG', 'TRATCONS', 'IDADE', 'SEXO', 'ECGRUP']
    data = df_target[features].copy()
    data = pd.get_dummies(data, columns=['ECGRUP'], drop_first=True, dtype=int)
    data = data.dropna()
    
    cph = CoxPHFitter()
    cph.fit(data, duration_col=duration_col, event_col=event_col)
    
    summary = cph.summary.copy()
    summary['HR'] = summary['exp(coef)'].round(2)
    summary['95% CI'] = summary.apply(
        lambda r: f"{r['exp(coef) lower 95%']:.2f}–{r['exp(coef) upper 95%']:.2f}", axis=1
    )
    summary['p-value'] = summary['p'].apply(lambda p: '< 0.001' if p < 0.001 else f"{p:.3f}")
    return cph, summary[['HR', '95% CI', 'z', 'p-value']]

cph_os, table_cox_os = fit_cox_model(df_surv_os, 'TIME_MONTHS_OS', 'EVENT_OS')
cph_rfs, table_cox_rfs = fit_cox_model(df_surv_rfs, 'TIME_MONTHS_RFS', 'EVENT_RECURRENCE')

print("=== Multivariable Cox Model: Overall Survival (OS) ===")
display(table_cox_os)

print("\n=== Multivariable Cox Model: Recurrence-Free Survival (RFS) ===")
display(table_cox_rfs)

## 6. Grid Search for Inflection Points
### 6.1 Formalism
The inflection point is identified by maximally selected log-rank statistics (Lausen and Schumacher, 1992) over an exhaustive daily grid of candidate cutpoints, rather than by arbitrary clinical thresholds:

$$Z^2(t) = \frac{\left(\sum_{i} \left[O_{1i} - E_{1i}\right]\right)^2}{\sum_{i} V_{1i}}$$

where
- $t$ is the candidate cutpoint separating Group 1 ($\le t$) from Group 2 ($> t$);
- $O_{1i}$ is the number of events observed in Group 1 at the $i$-th event time;
- $E_{1i}$ is the number of events expected in Group 1 under the null hypothesis of identical survival;
- $V_{1i}$ is the hypergeometric variance of $O_{1i} - E_{1i}$.

The inflection point $\tau$ is the cutpoint that maximizes $Z^2(t)$.

### 6.2 Search parameters
- Search window: 15 to 180 days, in 1-day steps.
- Minimum group size: 100 patients per group in the overall cohort and 50 per group in stage-specific analyses.
- Test: two-sample log-rank test (`lifelines.statistics.logrank_test`).

In [ ]:
# Section 6. Grid search for inflection points
def find_inflection_point(df_input, time_col, duration_col, event_col, min_day=15, max_day=180, min_n=100):
    """
    Exhaustive grid search for the waiting-time cutpoint that maximizes the log-rank statistic.

    For every candidate cutpoint d in [min_day, max_day] (1-day steps), patients are split into
    Group 1 (time_col <= d) and Group 2 (time_col > d). If both groups contain at least `min_n`
    patients, the two-sample log-rank test is computed. The cutpoint with the largest test
    statistic (Z^2) is returned as the inflection point.

    Parameters
    ----------
    df_input : pandas.DataFrame
        Cohort data.
    time_col : str
        Waiting-time column to be dichotomized (days), e.g. 'CONSDIAG', 'TRATCONS', 'DIAGTRAT'.
    duration_col : str
        Follow-up time column (months).
    event_col : str
        Event indicator column (1 = event, 0 = censored).
    min_day, max_day : int
        Limits of the search window (days).
    min_n : int
        Minimum number of patients required in each group.

    Returns
    -------
    dict or None
        'inflection_day', 'peak_z2', 'p_value' (nominal log-rank p-value at the optimum, not adjusted
        for the search), and the full grid ('all_days', 'all_scores'); None if no cutpoint is valid.
    """
    tested_days = np.arange(min_day, max_day + 1)
    scores = []
    p_vals = []
    valid_days = []
    
    for d in tested_days:
        g1 = df_input[df_input[time_col] <= d]
        g2 = df_input[df_input[time_col] > d]
        
        # Minimum group size, to keep the log-rank statistic stable
        if len(g1) >= min_n and len(g2) >= min_n:
            res = logrank_test(
                g1[duration_col], g2[duration_col],
                g1[event_col], g2[event_col]
            )
            scores.append(res.test_statistic)
            p_vals.append(res.p_value)
            valid_days.append(d)
            
    if not scores:
        return None
        
    best_idx = np.argmax(scores)
    return {
        'inflection_day': valid_days[best_idx],
        'peak_z2': scores[best_idx],
        'p_value': p_vals[best_idx],
        'all_days': valid_days,
        'all_scores': scores
    }

### 6.3 Inflection points in the overall cohort (OS and RFS)
The search is applied to $t_a$, $t_b$ and $t_c$ for each outcome. Each panel shows $Z^2$ as a function of the cutpoint; the dashed line marks the maximum.

In [ ]:
# Section 6.3. Overall-cohort inflection points (OS and RFS) for t_a, t_b and t_c
intervals = {
    'CONSDIAG': 't_a (Diagnosis to Consultation)',
    'TRATCONS': 't_b (Consultation to Treatment)',
    'DIAGTRAT': 't_c (Diagnosis to Treatment)'
}

fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey='row')
outcomes = [
    ('Overall Survival (OS)', df_surv_os, 'TIME_MONTHS_OS', 'EVENT_OS', 0, 'steelblue'),
    ('Recurrence-Free Survival (RFS)', df_surv_rfs, 'TIME_MONTHS_RFS', 'EVENT_RECURRENCE', 1, 'coral')
]

cohort_results = []

for outcome_name, data_df, dur_col, ev_col, row_idx, col_color in outcomes:
    for col_idx, (var_name, var_label) in enumerate(intervals.items()):
        res = find_inflection_point(data_df, var_name, dur_col, ev_col, min_day=15, max_day=180, min_n=100)
        ax = axes[row_idx, col_idx]
        
        if res:
            cohort_results.append({
                'Outcome': outcome_name,
                'Interval': var_label,
                'Inflection Cutpoint (Days)': res['inflection_day'],
                'Peak Z²': round(res['peak_z2'], 2),
                'p-value': '< 0.001' if res['p_value'] < 0.001 else f"{res['p_value']:.4f}"
            })
            
            ax.plot(res['all_days'], res['all_scores'], color=col_color, lw=2.2)
            ax.fill_between(res['all_days'], res['all_scores'], color=col_color, alpha=0.15)
            ax.axvline(res['inflection_day'], color='red', linestyle='--', lw=1.5)
            
            p_str = 'p < 0.001' if res['p_value'] < 0.001 else f"p = {res['p_value']:.3f}"
            ax.annotate(f"Cutoff: {res['inflection_day']} d\nZ²={res['peak_z2']:.1f}\n{p_str}",
                        xy=(res['inflection_day'], res['peak_z2']),
                        xytext=(res['inflection_day'] + 15, res['peak_z2'] * 0.85),
                        arrowprops=dict(arrowstyle='->', color='red'),
                        fontsize=10, weight='bold', color='red')
                        
        ax.set_title(f"{outcome_name} | {var_label}", fontsize=11, weight='bold')
        ax.set_xlabel('Cutoff Threshold (Days)')
        if col_idx == 0:
            ax.set_ylabel('Log-Rank Test Statistic (Z²)')

plt.tight_layout()
plt.show()

display(pd.DataFrame(cohort_results))

## 7. Stage-Specific Inflection Points and Multiplicity Adjustment
To assess whether tolerance to waiting depends on disease stage, the search is repeated within each AJCC stage (I–IV) for $t_a$ and $t_b$ and for both outcomes. The minimum group size is relaxed to $n \ge 50$ so that the search can cover early time windows with few events.

Raw p-values are adjusted for multiple comparisons across all stage-specific tests with the Benjamini–Hochberg procedure, yielding $q$-values. The raw p-values remain nominal with respect to the selection of the optimal cutpoint (see Statistical notes).

In [ ]:
# Section 7. Stage-specific inflection points with Benjamini-Hochberg FDR adjustment
stage_results = []
stages = ['I', 'II', 'III', 'IV']

for stage in stages:
    subset_os = df_surv_os[df_surv_os['ECGRUP'] == stage]
    subset_rfs = df_surv_rfs[df_surv_rfs['ECGRUP'] == stage]
    
    for var_name, var_label in [('CONSDIAG', 't_a'), ('TRATCONS', 't_b')]:
        # OS optimization
        res_os = find_inflection_point(subset_os, var_name, 'TIME_MONTHS_OS', 'EVENT_OS', min_day=15, max_day=180, min_n=50)
        if res_os:
            stage_results.append({
                'Survival Type': 'Overall Survival (OS)',
                'Variable': var_label,
                'Stage': stage,
                'Inflection Day': res_os['inflection_day'],
                'Peak Z²': round(res_os['peak_z2'], 2),
                'Raw p-value': res_os['p_value'],
                'N': len(subset_os)
            })
            
        # RFS optimization
        res_rfs = find_inflection_point(subset_rfs, var_name, 'TIME_MONTHS_RFS', 'EVENT_RECURRENCE', min_day=15, max_day=180, min_n=50)
        if res_rfs:
            stage_results.append({
                'Survival Type': 'Recurrence-Free Survival (RFS)',
                'Variable': var_label,
                'Stage': stage,
                'Inflection Day': res_rfs['inflection_day'],
                'Peak Z²': round(res_rfs['peak_z2'], 2),
                'Raw p-value': res_rfs['p_value'],
                'N': len(subset_rfs)
            })

df_stage_results = pd.DataFrame(stage_results)

# Benjamini-Hochberg FDR correction across all stage-specific tests
_, q_vals, _, _ = multipletests(df_stage_results['Raw p-value'], alpha=0.05, method='fdr_bh')
df_stage_results['q-value (FDR)'] = q_vals
df_stage_results['q-value (FDR)'] = df_stage_results['q-value (FDR)'].apply(lambda q: '< 0.001' if q < 0.001 else f"{q:.4f}")
df_stage_results['Raw p-value'] = df_stage_results['Raw p-value'].apply(lambda p: '< 0.001' if p < 0.001 else f"{p:.4f}")

# Stage-specific results (Table 3 of the manuscript)
display(df_stage_results.sort_values(by=['Survival Type', 'Variable', 'Stage']))

## 8. Export of Tables
Overall and stage-specific inflection points and the Cox model results are written to a single Excel workbook (`Inflection_Points_Manuscript_Tables.xlsx`), one sheet per table.

In [ ]:
# Section 8. Export of tables
# Export results to Excel (one sheet per table)
output_filename = 'Inflection_Points_Manuscript_Tables.xlsx'
with pd.ExcelWriter(output_filename) as writer:
    pd.DataFrame(cohort_results).to_excel(writer, sheet_name='Overall_Inflection', index=False)
    df_stage_results.to_excel(writer, sheet_name='Stage_Inflection', index=False)
    table_cox_os.to_excel(writer, sheet_name='Cox_OS')
    table_cox_rfs.to_excel(writer, sheet_name='Cox_RFS')

print(f"All manuscript tables successfully saved to '{output_filename}'!")